# Radial plots for UA and NWM at intermediary resolutions
---  
*J. Michelle Hu  
University of Utah  
June 2025

In [ ]:
import sys
from pathlib import PurePath, Path

import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
from rasterio.enums import Resampling

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc
import plot_sdd_radial as psr

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
def print_info_helper(ds):
    print(len(ds.x), len(ds.y), ds.rio.crs)

In [ ]:
diffdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'
var = 'depth'
resampling = 'nearest'
if resampling == 'average':
    resamp_alg = Resampling.average
elif resampling == 'cubic':
    resamp_alg = Resampling.cubic
elif resampling == 'bilinear':
    resamp_alg = Resampling.bilinear
else:
    resamp_alg = Resampling.nearest
for basin in ['animas', 'yampa', 'blue']:
    for WY in [2021, 2022, 2023, 2024]:
        for approach in ['ua', 'nwm']:
            print(f'Processing {basin} WY{WY} {approach} {var}')
            # Get the list of netCDF files
            nc_list = h.fn_list(diffdir, f'{basin}_wy{WY}_*{approach}*diff*{var}*original.nc')
            # if the list is empty, skip to the next in the loop
            if nc_list:
                for nc_fn in nc_list:
                    print(nc_fn)
                    dt = PurePath(nc_fn).stem.split('diff_')[1].split('_')[0]
                    runtype = PurePath(nc_fn).stem.split(f'wy{WY}_')[1].split('_')[0]
                    # Load the file
                    nc_ds = xr.open_dataset(nc_fn)
                    print_info_helper(nc_ds)

                    # Read in terrain data (elevation and aspect)
                    script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'
                    terrain_fn = h.fn_list(script_dir, f'{basin}*_setup/data/{basin}_terrain.nc')[0]
                    print(terrain_fn)

                    # Load the terrain dataset
                    ds = xr.open_dataset(terrain_fn, drop_variables=['spatial_ref', 'hs', 'slope'])
                    # Set the CRS
                    ds.rio.write_crs('epsg:32613', inplace=True)
                    print_info_helper(ds)

                    height = 5
                    figaspect_ratio = len(ds.x) / len(ds.y) # width to height ratio
                    fig, axa = plt.subplots(1, 5, figsize=(5*height*figaspect_ratio, height), sharex=True, sharey=True)
                    ds['dem'].plot.imshow(ax=axa[0], cbar_kwargs={'shrink':0.7})

                    # Do the same for the dataset
                    nc_ds = nc_ds.rio.reproject('EPSG:32613', resampling=resamp_alg)
                    print_info_helper(nc_ds)

                    nc_ds['depth_diff'].plot.imshow(ax=axa[1], cbar_kwargs={'shrink':0.7})

                    # Reproject the terrain and nc_ds dataset to an intermediary resolution
                    if approach == 'ua':
                        res = 400
                    elif approach == 'nwm':
                        res = 500

                    # Resample the datasets to the new resolution
                    ds_reproj = ds.rio.reproject('EPSG:32613', resolution=res, resampling=resamp_alg)
                    print_info_helper(ds_reproj)

                    ds_reproj['dem'].plot.imshow(ax=axa[2], cbar_kwargs={'shrink':0.7})

                    nc_ds_reproj = nc_ds.rio.reproject('EPSG:32613', resolution=res, resampling=resamp_alg)
                    print_info_helper(nc_ds_reproj)

                    nc_ds_reproj['depth_diff'].plot.imshow(ax=axa[3], cbar_kwargs={'shrink':0.7})

                    # Match the terrain to the dataset
                    ds_reproj = ds_reproj.rio.reproject_match(nc_ds_reproj, resampling=resamp_alg)
                    print_info_helper(ds_reproj)

                    ds_reproj['dem'].plot.imshow(ax=axa[4], cbar_kwargs={'shrink':0.7})
                    for ax in axa.flatten():
                        h.clean_axes(ax, gridon=False, fc='w')

                    # Generate a combined dataset
                    combo_ds = xr.merge([ds_reproj, nc_ds_reproj])

                    # Derive dem elevation ranges from the terrain dataset using hundreds place rounding and flexible bin number
                    p = None
                    _, dem_elev_ranges = proc.bin_elev(dem=combo_ds['dem'], basinname=basin,
                                                                plot_on=False, round_on=True, p=p, verbose=False)
                    # Extract elevation bins from the ranges
                    # Convert dict values into a flat list
                    elevation_bins = [f[0] for jdx, f in enumerate(dem_elev_ranges.values())]
                    # add the last bin (max elev)
                    elevation_bins.append(list(dem_elev_ranges.values())[-1][-1])

                    # Plot it up
                    num_aspect_bins = 8
                    aspect_labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
                    if p is not None:
                        outdir = f'/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/depth_diff/resampled_{resampling}/radial_{num_aspect_bins}_p{p}'
                    else:
                        outdir = f'/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/depth_diff/resampled_{resampling}/radial_{num_aspect_bins}'
                    # If directory doesn't exist, create it using pathlib
                    Path(outdir).mkdir(parents=True, exist_ok=True)
                    print(outdir)
                    if p is not None:
                        outname = f'{outdir}/{PurePath(nc_fn).stem.split('_original')[0]}_resamp{res}m_radial_{num_aspect_bins}_p{p}.png'
                    else:
                        outname = f'{outdir}/{PurePath(nc_fn).stem.split('_original')[0]}_resamp{res}m_radial_{num_aspect_bins}.png'
                    title = f'{basin.capitalize()} {runtype} {dt}'
                    cmap = sns.palettes.color_palette('PuOr', as_cmap=True)
                    psr.plot_radial(combo_ds=combo_ds, combo_var='depth_diff',
                                    combo_var_units='m',
                                    elevation_bins=elevation_bins,
                                    num_aspect_bins=num_aspect_bins,
                                    aspect_labels=aspect_labels,
                                    cmap=cmap, elev_fontcolor="k",
                                    outname=outname,
                                    title=title, vminnorm=-1, vmaxnorm=1,
                                    verbose=False)

In [ ]:
fig, axa = plt.subplots(3, 1, figsize=(4, 9))
ax = axa[0]
combo_ds['aspect'].plot.hist(bins=15, ec='k', ax=ax)
ax.set_title(f'{resampling.capitalize()} resampling')
ax.grid('gray', linestyle=':')
ax.set_ylim(0, 3500)

ax = axa[1]
combo_ds['depth_diff'].plot.hist(bins=25, ec='k', ax=ax)
ax.grid('gray', linestyle=':')
ax.set_title(f'{approach.upper()} - ASO')
ax.set_yscale('log')
ax.set_xlim(-1.25, 1.25)
ax.set_ylim(1, 1e4)

ax = axa[2]
combo_ds['dem'].plot.hist(bins=50, ec='k', ax=ax)
ax.grid('gray', linestyle=':')
ax.set_title('Elevation')
ax.set_xlim(1900, 4300)
ax.set_ylim(0, 1e3);
plt.tight_layout()

In [ ]:
ds = xr.open_dataset(terrain_fn, drop_variables=['spatial_ref', 'hs', 'slope'])
# Set the CRS
ds.rio.write_crs('epsg:32613', inplace=True)
nc_ds = xr.open_dataset(nc_fn)
# Set the CRS
ds.rio.write_crs('epsg:32613', inplace=True)


fig, axa = plt.subplots(3, 1, figsize=(4, 9))
ax = axa[0]
ds['aspect'].plot.hist(bins=15, ec='k', ax=ax)
ax.set_title('Original dataset')
ax.grid('gray', linestyle=':')
# ax.set_ylim(0, 3500)

ax = axa[1]
nc_ds['depth_diff'].plot.hist(bins=25, ec='k', ax=ax)
ax.grid('gray', linestyle=':')
ax.set_title(f'{approach.upper()} - ASO')
ax.set_yscale('log')
ax.set_xlim(-1.25, 1.25)
# ax.set_ylim(1, 1e4)

ax = axa[2]
ds['dem'].plot.hist(bins=50, ec='k', ax=ax)
ax.grid('gray', linestyle=':')
ax.set_title('Elevation')
ax.set_xlim(1900, 4300)
# ax.set_ylim(0, 1e3);
plt.tight_layout()